# 带时间窗的车辆路径问题 (CVRPTW)

**类别：** 路径规划

来源: [https://www.hexaly.com/templates/vehicle-routing-problem-with-time-windows-cvrptw](https://www.hexaly.com/templates/vehicle-routing-problem-with-time-windows-cvrptw)


## 问题

**在带时间窗的容量受限车辆路径问题 (CVRPTW) 中**，一组具有相同载重能力的配送车辆必须为客户提供服务。各客户具有已知的营业时间以及对单一商品的需求。车辆从同一个配送中心出发并最终返回该配送中心。每位客户必须在其营业时间内由恰好一辆车服务，且每辆车服务的需求总量不得超过其载重能力。优化目标是最小化所需车辆数以及总行驶距离。

	

### 学到的建模原则

- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模各卡车的客户访问序列
- 使用 [递归 lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 定义一个数组，以计算客户的访问时间
- 添加 [多目标](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html) 并将延迟（lateness）建模为 [软约束](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)


## 数据

所提供的带时间窗车辆路径问题 (CVRPTW) 实例来自 [Solomon 实例](http://web.cba.neu.edu/~msolomon/problems.htm)。数据文件的格式如下：

- 第一行：实例的名称
- 第五行：车辆数以及它们的共同载重能力
- 从第十行开始，每一行描述一位客户（从配送中心开始）：

- 客户的编号
- x 坐标
- y 坐标
- 需求量
- 最早到达时间
- 最晚到达时间
- 服务时间


## 模型

带时间窗容量受限车辆路径问题 (CVRPTW) 的 Hexaly 模型是 [CVRP 模型](https://www.hexaly.com/example/capacitated-vehicle-routing-problem-cvrp) 的一种扩展。问题中路径规划部分的细节（表示每辆卡车所分配客户的 list 决策变量、容量约束以及总行驶距离计算）请参考该模型。

由于时间窗较难严格满足，我们将其作为第一优先级目标处理，而非硬约束。如果卡车早于营业时间到达，则需等待客户开门；若晚于最晚到达时间到达，则度量并惩罚其延迟量。问题的第一个目标即为最小化所有客户的总延迟量。当该累计延迟量为零时，相应的解被认为是可行的。

我们使用 [递归数组](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 来计算每辆卡车对每位客户的访问结束时间。对每辆卡车和每位客户，到达时间取以下两者的最大值：

- 上一次访问的结束时间加上行驶时间（在所提供的实例中行驶时间即等于距离）。对于第一次访问，则是自配送中心出发的行驶时间（假设卡车在时刻 0 离开配送中心）。
- 该客户允许的最早到达时间。

结束时间即为该到达时间加上对该客户的服务时间。路径结束时返回配送中心的时刻为最后一次访问的结束时间加上从该处返回配送中心的行驶时间。根据这些结束时间，我们可以方便地计算累计延迟量。

最后，我们按字典序最小化总延迟量、所用车辆数以及总行驶距离。


## 结果

在 Solomon、Gehring 和 Homberger 研究基准上，Hexaly Optimizer 能够在 1 分钟运行时间内，使带时间窗容量受限车辆路径问题 (CVRPTW) 达到 1.8% 的平均最优性差距。我们的 [带时间窗容量受限车辆路径问题 (CVRPTW) 基准测试页面](https://www.hexaly.com/benchmark/hexaly-gurobi-or-tools-capacitated-vehicle-routing-problem-with-time-windows-cvrptw)展示了 Hexaly Optimizer 在这一具有挑战性的问题上如何超越 Gurobi 和 OR-Tools 等传统通用优化求解器。

[查看该基准测试](https://www.hexaly.com/benchmark/hexaly-gurobi-or-tools-capacitated-vehicle-routing-problem-with-time-windows-cvrptw)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def main(instance_file, str_time_limit, output_file):
    #
    # Read instance data
    #
    nb_customers, nb_trucks, truck_capacity, dist_matrix_data, dist_depot_data, \
        demands_data, service_time_data, earliest_start_data, latest_end_data, \
        max_horizon = read_input_cvrptw(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Sequence of customers visited by each truck
        customers_sequences = [model.list(nb_customers) for k in range(nb_trucks)]

        # All customers must be visited by exactly one truck
        model.constraint(model.partition(customers_sequences))

        # Create Hexaly arrays to be able to access them with an "at" operator
        demands = model.array(demands_data)
        earliest = model.array(earliest_start_data)
        latest = model.array(latest_end_data)
        service_time = model.array(service_time_data)
        dist_matrix = model.array(dist_matrix_data)
        dist_depot = model.array(dist_depot_data)

        dist_routes = [None] * nb_trucks
        end_time = [None] * nb_trucks
        home_lateness = [None] * nb_trucks
        lateness = [None] * nb_trucks

        # A truck is used if it visits at least one customer
        trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]
        nb_trucks_used = model.sum(trucks_used)

        for k in range(nb_trucks):
            sequence = customers_sequences[k]
            c = model.count(sequence)

            # The quantity needed in each route must not exceed the truck capacity
            demand_lambda = model.lambda_function(lambda j: demands[j])
            route_quantity = model.sum(sequence, demand_lambda)
            model.constraint(route_quantity <= truck_capacity)

            # Distance traveled by each truck
            dist_lambda = model.lambda_function(
                lambda i: model.at(dist_matrix, sequence[i - 1], sequence[i]))
            dist_routes[k] = model.sum(model.range(1, c), dist_lambda) \
                + model.iif(c > 0, dist_depot[sequence[0]] + dist_depot[sequence[c - 1]], 0)

            # End of each visit
            end_time_lambda = model.lambda_function(
                lambda i, prev:
                    model.max(
                        earliest[sequence[i]],
                        model.iif(
                            i == 0, dist_depot[sequence[0]],
                            prev + model.at(dist_matrix, sequence[i - 1], sequence[i])))
                    + service_time[sequence[i]])

            end_time[k] = model.array(model.range(0, c), end_time_lambda, 0)

            # Arriving home after max horizon
            home_lateness[k] = model.iif(
                trucks_used[k],
                model.max(
                    0,
                    end_time[k][c - 1] + dist_depot[sequence[c - 1]] - max_horizon),
                0)

            # Completing visit after latest end
            late_lambda = model.lambda_function(
                lambda i: model.max(0, end_time[k][i] - latest[sequence[i]]))
            lateness[k] = home_lateness[k] + model.sum(model.range(0, c), late_lambda)

        # Total lateness
        total_lateness = model.sum(lateness)

        # Total distance traveled
        total_distance = model.div(model.round(100 * model.sum(dist_routes)), 100)

        # Objective: minimize the number of trucks used, then minimize the distance traveled
        model.minimize(total_lateness)
        model.minimize(nb_trucks_used)
        model.minimize(total_distance)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - number of trucks used and total distance
        #  - for each truck the customers visited (omitting the start/end at the depot)
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d %d\n" % (nb_trucks_used.value, total_distance.value))
                for k in range(nb_trucks):
                    if trucks_used[k].value != 1:
                        continue
                    # Values in sequence are in 0...nbCustomers. +1 is to put it back in
                    # 1...nbCustomers+1 as in the data files (0 being the depot)
                    for customer in customers_sequences[k].value:
                        f.write("%d " % (customer + 1))
                    f.write("\n")


# The input files follow the "Solomon" format
def read_input_cvrptw(filename):
    file_it = iter(read_elem(filename))

    for i in range(4):
        next(file_it)

    nb_trucks = int(next(file_it))
    truck_capacity = int(next(file_it))

    for i in range(13):
        next(file_it)

    depot_x = int(next(file_it))
    depot_y = int(next(file_it))

    for i in range(2):
        next(file_it)

    max_horizon = int(next(file_it))

    next(file_it)

    customers_x = []
    customers_y = []
    demands = []
    earliest_start = []
    latest_end = []
    service_time = []

    while True:
        val = next(file_it, None)
        if val is None:
            break
        i = int(val) - 1
        customers_x.append(int(next(file_it)))
        customers_y.append(int(next(file_it)))
        demands.append(int(next(file_it)))
        ready = int(next(file_it))
        due = int(next(file_it))
        stime = int(next(file_it))
        earliest_start.append(ready)
        # in input files due date is meant as latest start time
        latest_end.append(due + stime)
        service_time.append(stime)

    nb_customers = i + 1

    # Compute distance matrix
    distance_matrix = compute_distance_matrix(customers_x, customers_y)
    distance_depots = compute_distance_depots(depot_x, depot_y, customers_x, customers_y)

    return nb_customers, nb_trucks, truck_capacity, distance_matrix, distance_depots, \
        demands, service_time, earliest_start, latest_end, max_horizon


# Computes the distance matrix
def compute_distance_matrix(customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_matrix = [[None for i in range(nb_customers)] for j in range(nb_customers)]
    for i in range(nb_customers):
        distance_matrix[i][i] = 0
        for j in range(nb_customers):
            dist = compute_dist(customers_x[i], customers_x[j],
                                customers_y[i], customers_y[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    return distance_matrix


# Computes the distances to depot
def compute_distance_depots(depot_x, depot_y, customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_depots = [None] * nb_customers
    for i in range(nb_customers):
        dist = compute_dist(depot_x, customers_x[i], depot_y, customers_y[i])
        distance_depots[i] = dist
    return distance_depots


def compute_dist(xi, xj, yi, yj):
    return math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2))


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python cvrptw.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instance_file, str_time_limit, output_file)
